In [7]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ClassicalRegister
from qiskit.circuit import QuantumRegister

LADDER_LENGTH = 5
TOTAL_QUBITS = 2 * LADDER_LENGTH

def a(i): return i
def b(i): return i + LADDER_LENGTH

qr = QuantumRegister(TOTAL_QUBITS, 'q')
cr = ClassicalRegister(LADDER_LENGTH - 2, 'c')
qc = QuantumCircuit(qr, cr)  # Both explicit Register objects

# 1. |+> on all qubits
for q in range(TOTAL_QUBITS):
    qc.h(q)
# 2. Top-row CZ gates
for i in range(LADDER_LENGTH - 1):
    qc.cz(a(i), a(i + 1))
# 3. Bottom-row
for i in range(LADDER_LENGTH - 1):
    qc.cz(b(i), b(i + 1))
# 4. Vertical rung
for i in range(LADDER_LENGTH):
    qc.cz(a(i), b(i))
# 5. Measure top-row qubits in X-basis
for i in range(1, LADDER_LENGTH - 1):
    qc.h(a(i))                      # Rotate to X-basis
    qc.measure(a(i), cr[i - 1])     # Measure into classical bit

    # Feed-forward: apply Z correction to output qubit if measurement = 1
    with qc.if_test((cr[i - 1], 1)):
        qc.z(a(LADDER_LENGTH - 1))

In [8]:
print(qc.draw(output='text', fold=-1))

     ┌───┐                                                                                                                        
q_0: ┤ H ├─■─────■────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├───┤ │     │       ┌───┐             ┌─┐                                                                                    
q_1: ┤ H ├─■──■──┼─────■─┤ H ├─────────────┤M├────────────────────────────────────────────────────────────────────────────────────
     ├───┤    │  │     │ └───┘   ┌───┐     └╥┘                          ┌─┐                                                       
q_2: ┤ H ├────■──┼──■──┼───────■─┤ H ├──────╫───────────────────────────┤M├───────────────────────────────────────────────────────
     ├───┤       │  │  │       │ └───┘┌───┐ ║                           └╥┘                          ┌─┐                          
q_3: ┤ H ├───────┼──■──┼───■───┼───■──┤ H ├─╫────────────────────────────╫─────────